# Zaid Hybrid IDS — Step 11G: ONNX Detector Runtime Optimization on Validation

This notebook closes the remaining initial-detector latency gap without retraining or changing the frozen random forest. The same trees are exported to ONNX and executed with a single-threaded CPU session optimized for one-flow inference.

Scientific safeguards:

- Uses all 120 Validation events only; Explainer Test is not defined or opened.
- Frozen trees, 60-feature order, threshold, TreeSHAP method, grounded renderer, and strict validator remain unchanged.
- Requires decision agreement, bounded probability difference, exact top-two SHAP features, exact explanation text, and 100% strict validity.
- Measures detector-only single-event latency separately from complete End-to-End latency.
- Accepts the deployment format only if detector P95 is below 10 ms and complete-path P95 is below 50 ms on Validation.


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)


Mounted at /content/drive


In [2]:
!pip install -q shap pyarrow skl2onnx onnxruntime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 48.6 MB/s eta 0:00:00


In [3]:
from pathlib import Path
from time import perf_counter_ns
from datetime import datetime, timezone
import hashlib
import json
import platform
import re
import sys

import joblib
import numpy as np
import pandas as pd
import shap
import sklearn
import onnxruntime as ort
import skl2onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

SEED = 20260831
THRESHOLD = 0.5
DETECTOR_TIMING_ROUNDS = 100
PIPELINE_TIMING_ROUNDS = 10
PROBABILITY_TOLERANCE = 1e-4
np.random.seed(SEED)

PROJECT_ROOT = Path('/content/drive/MyDrive/Zaid_Hybrid_IDS')
SPLIT_DIR = PROJECT_ROOT / 'data/splits'
EXPLAINER_DIR = PROJECT_ROOT / 'data/explainer/qwen25_15b_strict_repair_v1'
RESULT_ROOT = PROJECT_ROOT / 'results'
MODEL_ROOT = PROJECT_ROOT / 'models'

RAW_TEST_PATH = SPLIT_DIR / 'test.parquet'
VALIDATION_TARGET_PATH = EXPLAINER_DIR / 'teacher_targets_validation.jsonl'
FEATURE_MANIFEST_PATH = RESULT_ROOT / 'feature_sets_manifest.json'
SPLIT_MANIFEST_PATH = RESULT_ROOT / 'grouped_split_manifest.json'
FINAL_DETECTOR_MANIFEST_PATH = RESULT_ROOT / 'final_frozen_test_manifest.json'
FAST_PATH_VALIDATION_MANIFEST_PATH = RESULT_ROOT / 'grounded_fast_path_v2/grounded_fast_path_manifest.json'
RUNTIME_VALIDATION_MANIFEST_PATH = RESULT_ROOT / 'grounded_fast_path_runtime_optimization_v1/runtime_optimization_validation_manifest.json'
FROZEN_MODEL_PATH = MODEL_ROOT / 'random_forest_no_dst_port.joblib'
ONNX_MODEL_PATH = MODEL_ROOT / 'random_forest_no_dst_port_single_event_v1.onnx'
FINAL_RESULT_DIR = RESULT_ROOT / 'grounded_fast_path_onnx_runtime_optimization_v1'
FINAL_MANIFEST_PATH = FINAL_RESULT_DIR / 'onnx_runtime_validation_manifest.json'

for required_path in [
    RAW_TEST_PATH, VALIDATION_TARGET_PATH, FEATURE_MANIFEST_PATH,
    SPLIT_MANIFEST_PATH, FINAL_DETECTOR_MANIFEST_PATH,
    FAST_PATH_VALIDATION_MANIFEST_PATH, RUNTIME_VALIDATION_MANIFEST_PATH,
    FROZEN_MODEL_PATH,
]:
    assert required_path.exists(), f'Missing required frozen artifact: {required_path}'

assert not FINAL_MANIFEST_PATH.exists(), (
    f'ONNX runtime validation already completed: {FINAL_MANIFEST_PATH}'
)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def read_jsonl(path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

feature_manifest = read_json(FEATURE_MANIFEST_PATH)
split_manifest = read_json(SPLIT_MANIFEST_PATH)
detector_manifest = read_json(FINAL_DETECTOR_MANIFEST_PATH)
fast_validation_manifest = read_json(FAST_PATH_VALIDATION_MANIFEST_PATH)
runtime_validation_manifest = read_json(RUNTIME_VALIDATION_MANIFEST_PATH)

FEATURES = feature_manifest['no_dst_port_feature_set']
assert len(FEATURES) == 60
assert 'Dst Port' not in FEATURES
assert detector_manifest['threshold'] == THRESHOLD
assert detector_manifest['model_sha256'] == sha256_file(FROZEN_MODEL_PATH)
assert detector_manifest['test_file_sha256'] == split_manifest['outputs']['test']['sha256']
assert fast_validation_manifest['targets']['all_online_validation_targets_pass'] is True
assert runtime_validation_manifest['targets']['optimization_accepted'] is True

OPTIMIZATION_SPEC = {
    'version': '1.0',
    'scope': 'validation_only_120',
    'change': 'Export frozen RandomForestClassifier trees to ONNX Runtime for single-event classification',
    'source_model_sha256': detector_manifest['model_sha256'],
    'threshold': THRESHOLD,
    'feature_count': 60,
    'onnx_runtime_threads': 1,
    'probability_tolerance': PROBABILITY_TOLERANCE,
    'shap_method': 'TreeExplainer/tree_path_dependent/class_1/check_additivity_false',
    'renderer': 'grounded_fast_path_v2',
    'validator': 'strict_grounded_v2',
    'detector_latency_target_ms': 10.0,
    'pipeline_latency_target_ms': 50.0,
}
OPTIMIZATION_SPEC_SHA256 = hashlib.sha256(
    json.dumps(OPTIMIZATION_SPEC, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()

print('Frozen detector SHA-256:', detector_manifest['model_sha256'])
print('Frozen feature count:', len(FEATURES))
print('Frozen threshold:', THRESHOLD)
print('Optimization specification SHA-256:', OPTIMIZATION_SPEC_SHA256)
print('Model training or threshold tuning: False')
print('EXPLAINER TEST IS NOT DEFINED OR OPENED.')
print('ONNX RUNTIME OPTIMIZATION CONFIGURATION: FROZEN')


Frozen detector SHA-256: b0d0dd5ee8dafe1a0451958f1ac52247eb872217157e88631b83ae22fa59fe65
Frozen feature count: 60
Frozen threshold: 0.5
Optimization specification SHA-256: 1380e0c7685cad18ecac9010c752ff9179785f9f6bcfcf903b0b53a00b77473a
Model training or threshold tuning: False
EXPLAINER TEST IS NOT DEFINED OR OPENED.
ONNX RUNTIME OPTIMIZATION CONFIGURATION: FROZEN


In [4]:
IP_PATTERN = re.compile(r'(?<![\d.])(?:\d{1,3}\.){3}\d{1,3}(?![\d.])')
PROHIBITED_PATTERN = re.compile(
    r'(?i)\b(exploit|payload|reverse shell|credential stuffing tool|malware family|attacker identity)\b'
)
ATTACK_SUBTYPE_PATTERN = re.compile(
    r'(?i)\b(ftp|ssh|brute[ -]?force|botnet|ddos|dos|infiltration|sql injection|xss|heartbleed)\b'
)
KNOWN_FEATURE_NAMES = sorted(FEATURES, key=len, reverse=True)

def normalize_output(text):
    text = text.strip().replace('\r\n', '\n').replace('\r', '\n')
    return '\n'.join(line.strip() for line in text.split('\n') if line.strip())

def supports_detector_decision(feature, decision):
    supports_attack = feature['direction'] == 'supports_attack'
    decision_is_attack = decision.lower() == 'attack'
    return supports_attack == decision_is_attack

def evidence_sentence(first, second, decision):
    first_supports = supports_detector_decision(first, decision)
    second_supports = supports_detector_decision(second, decision)
    if first_supports and second_supports:
        return (
            f"{first['name']} and {second['name']} provided the strongest local "
            "SHAP support for the detector decision."
        )
    if first_supports and not second_supports:
        return (
            f"{first['name']} supported the detector decision, while "
            f"{second['name']} opposed it according to local SHAP evidence."
        )
    if not first_supports and second_supports:
        return (
            f"{second['name']} supported the detector decision, while "
            f"{first['name']} opposed it according to local SHAP evidence."
        )
    return (
        f"{first['name']} and {second['name']} locally opposed the detector "
        "decision according to SHAP evidence."
    )

def render_grounded_fast(record):
    first, second = record['top_features'][:2]
    decision = str(record['predicted_class'])
    confidence = float(record['confidence'])
    return (
        f"Summary: {decision} with {confidence:.3f} confidence\n"
        f"Evidence: {evidence_sentence(first, second, decision)}\n"
        "Action: Review network, authentication, and host logs, corroborate before containment."
    )

def strict_validate(record, text):
    normalized = normalize_output(text)
    lines = normalized.splitlines()
    schema = (
        len(lines) == 3
        and lines[0].startswith('Summary:')
        and lines[1].startswith('Evidence:')
        and lines[2].startswith('Action:')
    )
    first, second = record['top_features'][:2]
    summary = lines[0] if len(lines) > 0 else ''
    evidence = lines[1] if len(lines) > 1 else ''
    action = lines[2] if len(lines) > 2 else ''
    decision = str(record['predicted_class'])
    exact_confidence = f"{float(record['confidence']):.3f}"
    decision_aligned = decision.lower() in summary.lower()
    confidence_aligned = exact_confidence in summary
    required_features = first['name'] in evidence and second['name'] in evidence
    allowed_names = {first['name'], second['name']}
    unsupported = any(name in evidence and name not in allowed_names for name in KNOWN_FEATURE_NAMES)
    decision_support = {
        supports_detector_decision(first, decision),
        supports_detector_decision(second, decision),
    }
    evidence_lower = evidence.lower()
    if decision_support == {True}:
        direction_aligned = 'support' in evidence_lower
    elif decision_support == {False}:
        direction_aligned = 'oppos' in evidence_lower
    else:
        direction_aligned = 'support' in evidence_lower and 'oppos' in evidence_lower
    action_lower = action.lower()
    action_grounded = (
        'review' in action_lower
        and 'log' in action_lower
        and 'corroborat' in action_lower
        and 'containment' in action_lower
    )
    prohibited = bool(
        IP_PATTERN.search(normalized)
        or PROHIBITED_PATTERN.search(normalized)
        or ATTACK_SUBTYPE_PATTERN.search(normalized)
    )
    valid = all([
        schema, decision_aligned, confidence_aligned, required_features,
        direction_aligned, action_grounded, not unsupported, not prohibited,
    ])
    return {
        'normalized': normalized,
        'schema_compliant': schema,
        'decision_aligned': decision_aligned,
        'confidence_aligned': confidence_aligned,
        'required_features_present': required_features,
        'direction_aligned': direction_aligned,
        'action_grounded': action_grounded,
        'unsupported_feature': unsupported,
        'prohibited_content': prohibited,
        'valid': valid,
    }

def extract_attack_shap(raw_values):
    if isinstance(raw_values, list):
        array = np.asarray(raw_values[1])
    else:
        raw_array = np.asarray(raw_values)
        if raw_array.ndim == 3 and raw_array.shape[-1] == 2:
            array = raw_array[:, :, 1]
        elif raw_array.ndim == 3 and raw_array.shape[0] == 2:
            array = raw_array[1]
        elif raw_array.ndim == 2:
            array = raw_array
        else:
            raise ValueError(f'Unexpected SHAP shape: {raw_array.shape}')
    assert array.shape[1] == len(FEATURES), array.shape
    return array

print('FROZEN RENDERER, VALIDATOR, AND SHAP ADAPTER: READY')


FROZEN RENDERER, VALIDATOR, AND SHAP ADAPTER: READY


In [5]:
data_load_start_ns = perf_counter_ns()
raw_test_df = pd.read_parquet(
    RAW_TEST_PATH,
    columns=[*FEATURES, 'Label', 'flow_group_id'],
)
X_source_df = raw_test_df[FEATURES].astype(np.float32)
X_source_np = np.ascontiguousarray(X_source_df.to_numpy(copy=False), dtype=np.float32)
data_load_ms = (perf_counter_ns() - data_load_start_ns) / 1_000_000.0
assert raw_test_df.shape == (156717, 62), raw_test_df.shape
assert np.isfinite(X_source_np).all()

initialization_start_ns = perf_counter_ns()
frozen_model = joblib.load(FROZEN_MODEL_PATH)
stored_n_jobs = frozen_model.n_jobs
frozen_model.n_jobs = 1
tree_explainer = shap.TreeExplainer(
    frozen_model,
    feature_perturbation='tree_path_dependent',
)

onnx_graph = convert_sklearn(
    frozen_model,
    initial_types=[('flow_features', FloatTensorType([None, len(FEATURES)]))],
    options={id(frozen_model): {'zipmap': False}},
    target_opset=17,
)
onnx_bytes = onnx_graph.SerializeToString()

session_options = ort.SessionOptions()
session_options.intra_op_num_threads = 1
session_options.inter_op_num_threads = 1
session_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
ort_session = ort.InferenceSession(
    onnx_bytes,
    sess_options=session_options,
    providers=['CPUExecutionProvider'],
)
onnx_input_name = ort_session.get_inputs()[0].name
onnx_output_names = [item.name for item in ort_session.get_outputs()]
assert len(onnx_output_names) >= 2, onnx_output_names
initialization_ms = (perf_counter_ns() - initialization_start_ns) / 1_000_000.0

feature_array = np.asarray(FEATURES)

def onnx_attack_probability(row_np):
    outputs = ort_session.run(None, {onnx_input_name: row_np})
    probabilities = np.asarray(outputs[1])
    assert probabilities.shape == (1, 2), probabilities.shape
    return float(probabilities[0, 1])

def run_integrated_event(expected_record):
    total_start_ns = perf_counter_ns()
    source_row = int(expected_record['source_test_row'])
    row_np = X_source_np[source_row:source_row + 1]
    row_df = X_source_df.iloc[[source_row]]
    after_prepare_ns = perf_counter_ns()

    attack_probability = onnx_attack_probability(row_np)
    predicted_class = 'Attack' if attack_probability >= THRESHOLD else 'Benign'
    detector_confidence = attack_probability if predicted_class == 'Attack' else 1.0 - attack_probability
    after_detector_ns = perf_counter_ns()

    raw_shap = tree_explainer.shap_values(row_df, check_additivity=False)
    attack_shap = extract_attack_shap(raw_shap)[0]
    ordered_indices = np.argsort(np.abs(attack_shap))[::-1][:5]
    top_features = [
        {
            'name': str(feature_array[index]),
            'value': float(row_np[0, index]),
            'shap_value': float(attack_shap[index]),
            'direction': 'supports_attack' if attack_shap[index] > 0 else 'opposes_attack',
        }
        for index in ordered_indices
    ]
    after_shap_ns = perf_counter_ns()

    runtime_record = {
        'event_id': expected_record['event_id'],
        'source_test_row': source_row,
        'predicted_class': predicted_class,
        'confidence': detector_confidence,
        'top_features': top_features,
    }
    explanation = render_grounded_fast(runtime_record)
    after_render_ns = perf_counter_ns()
    check = strict_validate(runtime_record, explanation)
    after_validate_ns = perf_counter_ns()

    expected_names = [item['name'] for item in expected_record['top_features'][:2]]
    runtime_names = [item['name'] for item in top_features[:2]]
    expected_decision = str(expected_record['predicted_class'])
    expected_confidence = float(expected_record['confidence'])

    return {
        'event_id': expected_record['event_id'],
        'source_test_row': source_row,
        'predicted_class': predicted_class,
        'attack_probability': attack_probability,
        'detector_confidence': detector_confidence,
        'expected_confidence': expected_confidence,
        'decision_fidelity': predicted_class == expected_decision,
        'confidence_delta': abs(detector_confidence - expected_confidence),
        'confidence_fidelity': abs(detector_confidence - expected_confidence) <= PROBABILITY_TOLERANCE,
        'top2_feature_fidelity': runtime_names == expected_names,
        'runtime_top2_features': runtime_names,
        'expected_top2_features': expected_names,
        'explanation': check['normalized'],
        'strict_valid': check['valid'],
        'fallback_required': not check['valid'],
        'prepare_ms': (after_prepare_ns - total_start_ns) / 1_000_000.0,
        'detector_ms': (after_detector_ns - after_prepare_ns) / 1_000_000.0,
        'shap_and_selection_ms': (after_shap_ns - after_detector_ns) / 1_000_000.0,
        'renderer_ms': (after_render_ns - after_shap_ns) / 1_000_000.0,
        'validator_ms': (after_validate_ns - after_render_ns) / 1_000_000.0,
        'end_to_end_ms': (after_validate_ns - total_start_ns) / 1_000_000.0,
    }

validation_records = read_jsonl(VALIDATION_TARGET_PATH)
assert len(validation_records) == 120
assert all(record['split'] == 'validation' for record in validation_records)

# Numerical equivalence gate before timing.
source_rows = np.asarray([int(record['source_test_row']) for record in validation_records])
validation_np = X_source_np[source_rows]
sklearn_probabilities = frozen_model.predict_proba(validation_np)[:, 1]
onnx_probabilities = np.asarray(
    ort_session.run(None, {onnx_input_name: validation_np})[1]
)[:, 1]
max_probability_delta = float(np.max(np.abs(sklearn_probabilities - onnx_probabilities)))
decision_agreement = bool(np.array_equal(
    sklearn_probabilities >= THRESHOLD,
    onnx_probabilities >= THRESHOLD,
))
assert max_probability_delta <= PROBABILITY_TOLERANCE, max_probability_delta
assert decision_agreement

zero_row = np.zeros((1, len(FEATURES)), dtype=np.float32)
for _ in range(20):
    _ = onnx_attack_probability(zero_row)
    _ = frozen_model.predict_proba(zero_row)

print('Raw source rows:', len(raw_test_df))
print('Offline data load ms:', data_load_ms)
print('Offline model, TreeExplainer, ONNX conversion and session initialization ms:', initialization_ms)
print('ONNX input:', onnx_input_name)
print('ONNX outputs:', onnx_output_names)
print('Validation events:', len(validation_records))
print('Maximum sklearn-to-ONNX probability delta:', max_probability_delta)
print('Validation decision agreement:', decision_agreement)
print('EXPLAINER TEST IS NOT DEFINED OR OPENED.')
print('ONNX OPTIMIZATION DATA: READY')


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local

Raw source rows: 156717
Offline data load ms: 2731.279697
Offline model, TreeExplainer, ONNX conversion and session initialization ms: 8069.039836
ONNX input: flow_features
ONNX outputs: ['label', 'probabilities']
Validation events: 120
Maximum sklearn-to-ONNX probability delta: 7.836539375194462e-07
Validation decision agreement: True
EXPLAINER TEST IS NOT DEFINED OR OPENED.
ONNX OPTIMIZATION DATA: READY


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [6]:
# Detector-only single-event timing. Each call includes row slicing, inference,
# probability extraction, thresholding, and confidence calculation.
detector_timing_rows = []
for timing_round in range(1, DETECTOR_TIMING_ROUNDS + 1):
    for record in validation_records:
        total_start_ns = perf_counter_ns()
        source_row = int(record['source_test_row'])
        row_np = X_source_np[source_row:source_row + 1]
        probability = onnx_attack_probability(row_np)
        decision = 'Attack' if probability >= THRESHOLD else 'Benign'
        confidence = probability if decision == 'Attack' else 1.0 - probability
        total_end_ns = perf_counter_ns()
        detector_timing_rows.append({
            'event_id': record['event_id'],
            'timing_round': timing_round,
            'decision': decision,
            'confidence': confidence,
            'detector_end_to_end_ms': (total_end_ns - total_start_ns) / 1_000_000.0,
        })
    if timing_round % 20 == 0:
        print(f'Completed detector-only round {timing_round}/{DETECTOR_TIMING_ROUNDS}')

detector_timing_df = pd.DataFrame(detector_timing_rows)
detector_values = detector_timing_df['detector_end_to_end_ms'].to_numpy(dtype=np.float64)
detector_summary = {
    'calls': int(len(detector_values)),
    'mean_ms': float(detector_values.mean()),
    'p50_ms': float(np.quantile(detector_values, 0.50)),
    'p95_ms': float(np.quantile(detector_values, 0.95)),
    'p99_ms': float(np.quantile(detector_values, 0.99)),
    'max_ms': float(detector_values.max()),
}

# Complete alert path timing with the unchanged TreeSHAP, renderer and validator.
pipeline_rows = []
first_round_results = []
for timing_round in range(1, PIPELINE_TIMING_ROUNDS + 1):
    round_results = [run_integrated_event(record) for record in validation_records]
    if timing_round == 1:
        first_round_results = round_results
    else:
        assert all(
            current['explanation'] == baseline['explanation']
            for current, baseline in zip(round_results, first_round_results)
        )
    for result in round_results:
        pipeline_rows.append({'timing_round': timing_round, **result})
    print(f'Completed full-pipeline round {timing_round}/{PIPELINE_TIMING_ROUNDS}')

event_df = pd.DataFrame(first_round_results)
pipeline_timing_df = pd.DataFrame(pipeline_rows)
components = [
    'prepare_ms', 'detector_ms', 'shap_and_selection_ms',
    'renderer_ms', 'validator_ms', 'end_to_end_ms',
]
summary_rows = []
for component in components:
    values = pipeline_timing_df[component].to_numpy(dtype=np.float64)
    summary_rows.append({
        'component': component,
        'calls': int(len(values)),
        'mean_ms': float(values.mean()),
        'p50_ms': float(np.quantile(values, 0.50)),
        'p95_ms': float(np.quantile(values, 0.95)),
        'p99_ms': float(np.quantile(values, 0.99)),
        'max_ms': float(values.max()),
    })
pipeline_summary_df = pd.DataFrame(summary_rows)
end_to_end_summary = pipeline_summary_df.loc[
    pipeline_summary_df['component'].eq('end_to_end_ms')
].iloc[0]

quality = {
    'strict_valid_rate': float(event_df['strict_valid'].mean()),
    'fallback_rate': float(event_df['fallback_required'].mean()),
    'decision_fidelity_rate': float(event_df['decision_fidelity'].mean()),
    'confidence_fidelity_rate': float(event_df['confidence_fidelity'].mean()),
    'top2_feature_fidelity_rate': float(event_df['top2_feature_fidelity'].mean()),
    'max_expected_confidence_delta': float(event_df['confidence_delta'].max()),
}
detector_target_pass = detector_summary['p95_ms'] < 10.0
pipeline_target_pass = float(end_to_end_summary['p95_ms']) < 50.0
all_targets_pass = bool(
    detector_target_pass
    and pipeline_target_pass
    and quality['strict_valid_rate'] == 1.0
    and quality['fallback_rate'] == 0.0
    and quality['decision_fidelity_rate'] == 1.0
    and quality['confidence_fidelity_rate'] == 1.0
    and quality['top2_feature_fidelity_rate'] == 1.0
)

print('\nDetector-only latency summary:')
print(json.dumps(detector_summary, indent=2))
print('\nIntegrated pipeline latency summary:')
print(pipeline_summary_df.to_string(index=False))
print('\nQuality:', json.dumps(quality, indent=2))
print('Detector P95 <10 ms:', detector_target_pass)
print('End-to-End P95 <50 ms:', pipeline_target_pass)
print('ALL ONNX VALIDATION TARGETS PASS:', all_targets_pass)
assert all_targets_pass
print('ONNX RUNTIME VALIDATION: COMPLETE')


Completed detector-only round 20/100
Completed detector-only round 40/100
Completed detector-only round 60/100
Completed detector-only round 80/100
Completed detector-only round 100/100
Completed full-pipeline round 1/10
Completed full-pipeline round 2/10
Completed full-pipeline round 3/10
Completed full-pipeline round 4/10
Completed full-pipeline round 5/10
Completed full-pipeline round 6/10
Completed full-pipeline round 7/10
Completed full-pipeline round 8/10
Completed full-pipeline round 9/10
Completed full-pipeline round 10/10

Detector-only latency summary:
{
  "calls": 12000,
  "mean_ms": 0.09783910208333332,
  "p50_ms": 0.036156999999999995,
  "p95_ms": 0.08703489999999994,
  "p99_ms": 2.195388020000002,
  "max_ms": 9.252367
}

Integrated pipeline latency summary:
            component  calls  mean_ms   p50_ms    p95_ms    p99_ms    max_ms
           prepare_ms   1200 0.413384 0.314312  0.475495  4.097727  8.512669
          detector_ms   1200 0.183637 0.162404  0.228068  0.4269

In [7]:
FINAL_RESULT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_MODEL_PATH.write_bytes(onnx_bytes)

DETECTOR_TIMING_PATH = FINAL_RESULT_DIR / 'onnx_detector_validation_timing_calls.parquet'
PIPELINE_TIMING_PATH = FINAL_RESULT_DIR / 'onnx_pipeline_validation_timing_calls.parquet'
PIPELINE_SUMMARY_PATH = FINAL_RESULT_DIR / 'onnx_pipeline_validation_latency_summary.csv'
EVENT_RESULTS_PATH = FINAL_RESULT_DIR / 'onnx_validation_event_results.parquet'

detector_timing_df.to_parquet(DETECTOR_TIMING_PATH, index=False, compression='zstd')
pipeline_timing_df.to_parquet(PIPELINE_TIMING_PATH, index=False, compression='zstd')
pipeline_summary_df.to_csv(PIPELINE_SUMMARY_PATH, index=False)
event_df.to_parquet(EVENT_RESULTS_PATH, index=False, compression='zstd')

manifest = {
    'schema_version': '1.0',
    'stage': 'validation_only_onnx_detector_runtime_optimization',
    'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
    'optimization_specification': OPTIMIZATION_SPEC,
    'optimization_specification_sha256': OPTIMIZATION_SPEC_SHA256,
    'source_model_file': str(FROZEN_MODEL_PATH),
    'source_model_sha256': detector_manifest['model_sha256'],
    'onnx_model_file': str(ONNX_MODEL_PATH),
    'onnx_model_sha256': sha256_file(ONNX_MODEL_PATH),
    'validation_events': len(validation_records),
    'detector_timing_rounds': DETECTOR_TIMING_ROUNDS,
    'pipeline_timing_rounds': PIPELINE_TIMING_ROUNDS,
    'max_sklearn_to_onnx_probability_delta': max_probability_delta,
    'decision_agreement': decision_agreement,
    'quality': quality,
    'detector_only_latency': detector_summary,
    'pipeline_latency': {
        row['component']: {
            key: int(row[key]) if key == 'calls' else float(row[key])
            for key in ['calls', 'mean_ms', 'p50_ms', 'p95_ms', 'p99_ms', 'max_ms']
        }
        for row in summary_rows
    },
    'targets': {
        'detector_single_event_p95_under_10ms': detector_target_pass,
        'integrated_single_event_p95_under_50ms': pipeline_target_pass,
        'zero_fallbacks': quality['fallback_rate'] == 0.0,
        'all_onnx_validation_targets_pass': all_targets_pass,
    },
    'test_dataset_loaded': False,
    'model_training_performed': False,
    'threshold_tuning_performed': False,
    'software': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'sklearn': sklearn.__version__,
        'shap': shap.__version__,
        'skl2onnx': skl2onnx.__version__,
        'onnxruntime': ort.__version__,
    },
    'outputs': {
        'detector_timing_calls': str(DETECTOR_TIMING_PATH),
        'pipeline_timing_calls': str(PIPELINE_TIMING_PATH),
        'pipeline_latency_summary': str(PIPELINE_SUMMARY_PATH),
        'event_results': str(EVENT_RESULTS_PATH),
    },
}

with FINAL_MANIFEST_PATH.open('x', encoding='utf-8') as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)

print('Saved ONNX model:', ONNX_MODEL_PATH)
print('Saved:', DETECTOR_TIMING_PATH)
print('Saved:', PIPELINE_TIMING_PATH)
print('Saved:', PIPELINE_SUMMARY_PATH)
print('Saved:', EVENT_RESULTS_PATH)
print('Saved:', FINAL_MANIFEST_PATH)
print('ALL ONNX VALIDATION TARGETS PASS:', all_targets_pass)
print('ONNX RUNTIME VALIDATION: SAVED AND FROZEN')


Saved ONNX model: /content/drive/MyDrive/Zaid_Hybrid_IDS/models/random_forest_no_dst_port_single_event_v1.onnx
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/grounded_fast_path_onnx_runtime_optimization_v1/onnx_detector_validation_timing_calls.parquet
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/grounded_fast_path_onnx_runtime_optimization_v1/onnx_pipeline_validation_timing_calls.parquet
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/grounded_fast_path_onnx_runtime_optimization_v1/onnx_pipeline_validation_latency_summary.csv
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/grounded_fast_path_onnx_runtime_optimization_v1/onnx_validation_event_results.parquet
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/grounded_fast_path_onnx_runtime_optimization_v1/onnx_runtime_validation_manifest.json
ALL ONNX VALIDATION TARGETS PASS: True
ONNX RUNTIME VALIDATION: SAVED AND FROZEN


In [8]:
for result in first_round_results[:6]:
    print('=' * 80)
    print('Event:', result['event_id'])
    print('Decision:', result['predicted_class'], '| Confidence:', f"{result['detector_confidence']:.6f}")
    print('Strict valid:', result['strict_valid'], '| End-to-End ms:', f"{result['end_to_end_ms']:.6f}")
    print(result['explanation'])
print('ONNX RUNTIME VALIDATION HUMAN QA EXAMPLES: REVIEWED')


Event: test-002432
Decision: Attack | Confidence: 0.999780
Strict valid: True | End-to-End ms: 42.217352
Summary: Attack with 1.000 confidence
Evidence: Fwd Seg Size Min and Init Fwd Win Byts provided the strongest local SHAP support for the detector decision.
Action: Review network, authentication, and host logs, corroborate before containment.
Event: test-003434
Decision: Attack | Confidence: 0.999782
Strict valid: True | End-to-End ms: 38.241792
Summary: Attack with 1.000 confidence
Evidence: Fwd Seg Size Min and Init Fwd Win Byts provided the strongest local SHAP support for the detector decision.
Action: Review network, authentication, and host logs, corroborate before containment.
Event: test-003680
Decision: Attack | Confidence: 0.999782
Strict valid: True | End-to-End ms: 30.345576
Summary: Attack with 1.000 confidence
Evidence: Fwd Seg Size Min and Init Fwd Win Byts provided the strongest local SHAP support for the detector decision.
Action: Review network, authentication, and

## Interpretation

The accepted change is an execution-only setting for single-flow inference. It does not alter the trained trees, probabilities, threshold, SHAP values, explanation text, or validator. The confirmatory Test result already reported in Step 11D remains unchanged; this Validation result supports a faster deployment configuration and a future independent operational benchmark.
